In [1]:
!pip uninstall -y torchvision -q
!pip install -q -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 143.7 MB/s eta 0:00:00


In [2]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)

model = model.to(device)

model.eval()

print("Model:", MODEL_NAME)
print("Device:", device)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

HTTP Error 429 thrown while requesting HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/7ae557604adf67be50417f59c2c2f167def9a775/vocab.json
Retrying in 1s [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/7ae557604adf67be50417f59c2c2f167def9a775/vocab.json
Retrying in 2s [Retry 2/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/7ae557604adf67be50417f59c2c2f167def9a775/vocab.json
Retrying in 4s [Retry 3/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/7ae557604adf67be50417f59c2c2f167def9a775/vocab.json
Retrying in 8s [Retry 4/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/7ae557604adf67be50417f59c2c2f167def9a775/vocab.json
Retrying in 8s [Retry 5/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instr

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

HTTP Error 500 thrown while requesting HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/7ae557604adf67be50417f59c2c2f167def9a775/tokenizer.json
Retrying in 1s [Retry 1/5].
HTTP Error 500 thrown while requesting HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/7ae557604adf67be50417f59c2c2f167def9a775/tokenizer.json
Retrying in 2s [Retry 2/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/7ae557604adf67be50417f59c2c2f167def9a775/tokenizer.json
Retrying in 4s [Retry 3/5].


...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model: Qwen/Qwen2.5-0.5B-Instruct
Device: cuda


In [3]:
def summarize_text(
    text,
    instruction="Metni kısa ve anlaşılır şekilde özetle.",
    max_new_tokens=150
):

    messages = [
        {
            "role": "system",
            "content": (
                "Sen teknik metinleri özetleyen yardımcı bir asistansın. "
                "Yalnızca verilen metindeki bilgileri kullan."
            )
        },
        {
            "role": "user",
            "content": f"""
Görev:
{instruction}

Metin:
{text}
"""
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[-1]:
    ]

    summary = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return summary

In [4]:
text = """
Elektrik motorlarında rulmanlar, rotorun düzgün ve düşük
sürtünmeyle dönmesini sağlayan kritik mekanik bileşenlerdir.
Rulman arızalarının en yaygın nedenleri arasında yetersiz
yağlama, yanlış hizalama, aşırı mekanik yük, kirlenme ve
hatalı montaj bulunmaktadır. Rulmanda meydana gelen aşınma
titreşim ve sıcaklık artışına neden olabilir. Arızanın erken
tespit edilmemesi motorun performansının düşmesine ve
ilerleyen durumlarda motorun tamamen durmasına yol açabilir.
Titreşim analizi, sıcaklık ölçümü ve akustik izleme gibi
kestirimci bakım yöntemleri rulman arızalarının erken
tespitinde kullanılmaktadır.
"""

In [5]:
summary = summarize_text(text)

print(summary)

Elektrik motorlarında rulmanlar, rotorun düzgün ve düşük sürtünmeyle dönmesini sağlayan kritik mekanik bileşenlerdir. Rulman arızalarının en yaygın nedenleri arasında yetersiz yağı, yanlış hizalama, aşırı mekanik yük, kirlenme ve hatalı montaj bulunmaktadır. Arızanın erken tesit edilmemesi motorun performansının düşmesine ve ilerleyen durumlarda motorun tamamen durmasına yol açabilir. Titreşim analizi ve sıcaklık ölçümü gibi kestirmiçi bakım yöntemleri rulman arızalarının erken


In [6]:
print(
    summarize_text(
        text,
        instruction="Bu metni yalnızca tek cümleyle özetle."
    )
)

Elektrik motorlarında rulmanlar, rotorun düzgün ve düşük sürtünmeyle dönmesini sağlayan kritik mekanik bileşenlerdir. Rulman arızalarının en yaygın nedenleri arasında yetersiz yağlama, yanlış hizalama, aşırı mekanik yük, kirlenme ve hatalı montaj bulunmaktadır. Arızanın erken tespit edilmemesi motorun performansının düşmesine ve ilerleyen durumlarda motorun tamamen durmasına yol açabilir. Titreşim analizi, sıcaklık ölçümü ve akustik izleme gibi kestirimci bakım yöntemleri rul


In [7]:
print(
    summarize_text(
        text,
        instruction=(
            "Metindeki en önemli bilgileri "
            "3 maddelik bir liste halinde özetle."
        )
    )
)

1. Elektrik motorlarında rulmanlar, rotorun düzgün ve düşük sürtünmeyle dönmesini sağlayan kritik mekanik bileşenlerdir.
2. Rulman arızalarının en yaygın nedenleri arasında yetersiz yağlama, yanlış hizalama, aşırı mekanik yük, kirlenme ve hatalı montaj bulunmaktadır.
3. Rulmanda meydana gelen aşınma titreşim ve sıcaklık artışına neden olabilir. Arızanın erken tespit edilmemesi motorun performansının düşmesine ve ilerleyen durumlarda motorun tamamen durmasına yol açabilir


In [8]:
print(
    summarize_text(
        text,
        instruction=(
            "Bir elektrik-elektronik mühendisi için "
            "kısa teknik özet hazırla."
        )
    )
)

Elektrik motorlarında rulmanlar, rotorun düzgün ve düşük sürtünmeyle dönmesini sağlayan kritik mekanik bileşenlerdir. Rulman arızalarının en yaygın nedenleri arasında yetersiz yağlama, yanlış hizalama, aşırı mekanik yük, kirlenme ve hatalı montaj bulunmaktadır. Rulmanda meydana gelen aşınma titreşim ve sıcaklık artışına neden olabilir. Arızanın erken tespit edilmemesi motorun performansının düşmesine ve ilerleyen durumlarda motorun tamamen durmasına yol açabilir. Titreşim analizi


In [9]:
print(
    summarize_text(
        text,
        instruction=(
            "Konuyu teknik bilgisi olmayan birine "
            "2-3 cümlede basitçe açıkla."
        )
    )
)

Elektrik motorlarında rulmanlar, rotorun düzgün ve düşük sürtünmeyle dönmesini sağlayan kritik mekanik bileşenlerdir. Rulman arızalarının en yaygın nedenleri arasında yetersiz yağlama, yanlış hizalama, aşırı mekanik yük, kirlenme ve hatalı montaj bulunmaktadır. Rulmanda meydana gelen aşınma titreşim ve sıcaklık artışına neden olabilir. Arızanın erken tespit edilmemesi motorun performansının düşmesine ve ilerleyen durumlarda motorun tamamen durmasına yol açabilir. Titreşim analizi


In [10]:
input_tokens = tokenizer.encode(
    text,
    add_special_tokens=False
)

print("Girdi token sayısı:", len(input_tokens))

Girdi token sayısı: 197


In [11]:
summary = summarize_text(
    text,
    instruction="Metni 3 cümlede özetle."
)

In [12]:
summary_tokens = tokenizer.encode(
    summary,
    add_special_tokens=False
)

print("Özet token sayısı:", len(summary_tokens))

Özet token sayısı: 150


In [13]:
compression_ratio = (
    len(summary_tokens)
    /
    len(input_tokens)
)

print(
    "Sıkıştırma oranı:",
    round(compression_ratio, 3)
)

Sıkıştırma oranı: 0.761
